# 1. Import & Setup

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# 2. Daten einlesen

In [40]:

df_raw = pd.read_csv(
    "../data/Number_of_fires_by_month.csv",
    sep=";",          # falls nötig
    skiprows=1        # falls Kopfzeile erklärt wird
)

df_raw.head()

,Jurisdiction,Month,Data Qualifier,1990,1991,1992,1993,1994,1995,1996,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Alberta,January,a,1.0,9.0,14.0,8.0,4.0,10.0,1.0,...,9.0,2.0,7.0,4.0,4.0,17.0,1.0,8.0,1.0,NaN
1,Alberta,February,a,5.0,4.0,11.0,12.0,NaN,6.0,NaN,...,NaN,4.0,12.0,4.0,NaN,1.0,1.0,1.0,6.0,NaN
2,Alberta,March,a,8.0,8.0,61.0,29.0,12.0,29.0,1.0,...,11.0,32.0,62.0,14.0,19.0,34.0,4.0,34.0,21.0,NaN
3,Alberta,April,a,26.0,111.0,110.0,52.0,64.0,57.0,26.0,...,81.0,277.0,287.0,101.0,97.0,188.0,68.0,153.0,132.0,NaN
4,Alberta,May,a,114.0,201.0,91.0,242.0,132.0,215.0,56.0,...,360.0,505.0,250.0,348.0,454.0,315.0,209.0,282.0,226.0,NaN


# 3. Unnötige Spalte entfernen

In [41]:
df = df_raw.drop(columns=["Data Qualifier"])

# 4. Wide to Long

In [42]:
df_long = df.melt(
    id_vars=["Jurisdiction", "Month"],
    var_name="Year",
    value_name="Number_fires"
)

df_long["Year"] = df_long["Year"].astype(int)

df_long["Jurisdiction"] = df_long["Jurisdiction"].str.strip()

df_long["Number_fires"] = (
    pd.to_numeric(df_long["Number_fires"], errors="coerce")
    .fillna(0)
    .astype(int)
)

month_map = {
    "January": 1, "February": 2, "March": 3,
    "April": 4, "May": 5, "June": 6,
    "July": 7, "August": 8, "September": 9,
    "October": 10, "November": 11, "December": 12
}

df_long["Month"] = df_long["Month"].map(month_map)

df_long = df_long[
    df_long["Month"].notna() &
    df_long["Month"].between(1, 12)
]

df_long = df_long.sort_values(
    ["Jurisdiction", "Year", "Month"]
)

print("RAW:", df_raw.columns.tolist())
print("LONG:", df_long.columns.tolist())
df_long.tail(5)

RAW: ['Jurisdiction', 'Month', 'Data Qualifier', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023']
LONG: ['Jurisdiction', 'Month', 'Year', 'Number_fires']


,Jurisdiction,Month,Year,Number_fires
5433,Yukon,8.0,2023,0
5434,Yukon,9.0,2023,0
5435,Yukon,10.0,2023,0
5436,Yukon,11.0,2023,0
5437,Yukon,12.0,2023,0


# 6. 2023 raus

In [43]:
df_long = df_long[df_long["Year"] <= 2022]

# 5. Neu speichern

In [44]:
df_long[["Year", "Month", "Jurisdiction", "Number_fires"]].to_csv(
    "../data/fires_history_all_clean.csv",
    index=False
)